# 09 DL Prediction

Load the saved DL checkpoint, split supervised windows 80:20, pick random test windows, predict with the multimodal model, and save the prediction table.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import ARCHIVE_DIR, ALIGNED_DIR, FEATURES_DIR, MODELS_DIR, OUTPUT_DIR
from src.data.archive_scanner import scan_archive
from src.models.datasets import add_window_targets, create_window_index
from src.models.prediction import predict_dl_samples, random_sample, split_80_20


## 1. Load features, windows, and checkpoint

In [ ]:
features = pd.read_csv(FEATURES_DIR / 'trial_features.csv')
windows_path = FEATURES_DIR / 'dl_windows.csv'
if windows_path.exists():
    windows = pd.read_csv(windows_path)
else:
    records = scan_archive(ARCHIVE_DIR)
    windows = add_window_targets(create_window_index(ALIGNED_DIR, archive_records=records), features)
    windows.to_csv(windows_path, index=False)

checkpoint_path = MODELS_DIR / 'multimodal_physio_net.pt'
windows.shape, checkpoint_path


## 2. Split 80:20 and sample from test windows

In [ ]:
train_windows, test_windows = split_80_20(windows, random_state=42, stratify_col='cv_load_class')
samples = random_sample(test_windows, n=8, random_state=7)
train_windows.shape, test_windows.shape, samples[['subject_id', 'trial_id', 'start_row', 'end_row', 'hr_target', 'cv_load_class', 'bp_proxy_target']]


## 3. Predict and save results

In [ ]:
# use_video=False matches the fast debug checkpoint training path. Set True only after training a checkpoint with real video clips.
predictions = predict_dl_samples(
    samples,
    feature_table=features,
    checkpoint_path=checkpoint_path,
    sequence_length=64,
    frame_size=32,
    video_frames=8,
    use_video=False,
    batch_size=2,
    device='cpu',
)
pred_dir = OUTPUT_DIR / 'predictions'
pred_dir.mkdir(parents=True, exist_ok=True)
prediction_path = pred_dir / 'dl_random_sample_predictions.csv'
predictions.to_csv(prediction_path, index=False)
prediction_path, predictions
